
# GPT-J-6B ROME Reproduction - One-Click Google Colab Notebook

Reproduce ROME (Rank-One Model Editing) on **GPT-J-6B** for single-fact editing using the EasyEdit framework (CISC7021 Group26 W2 task).

**How to run**: Menu Runtime -> Run all. Cells are ordered by dependency and executed from top to bottom.

**Deliverables**: editing evaluation metrics (reliability / generalisation / locality / portability) are written to a JSON file in the Colab root directory, optionally saved to Google Drive or downloaded directly.

## Workflow Overview

| Step | Content | Estimated Time |
|---|---|---|
| 1 | Clone project repo + install environment | 3-8 min |
| 2 | Download GPT-J-6B fp16 weights (~11.2GB) | 5-20 min |
| 3 | Run ROME editing (zsRE subset) | 15-60 min |
| 4 | Save results (Drive / browser download) | <1 min |

> Note: keep the browser tab open during execution; after a disconnect you can Run All again (the cached model is automatically skipped).



## 1. Environment Setup

Colab ships with a CUDA-enabled PyTorch, but EasyEdit's 2026 requirements pin versions such as `torch==2.9.1`, `transformers==5.5.4`, `huggingface_hub==1.17.0`.

Steps:
1. Clone the project repository (contains EasyEdit and the W2 reproduction script `EasyEdit/examples/w2_rome_repro.py`);
2. Check the torch version; if it is not 2.9.1, install the CUDA build `torch==2.9.1` from `download.pytorch.org/whl/cu126`;
3. Install the remaining dependencies (`pip install -r requirements.txt` skips already-installed packages, with a fallback install for missing key packages);
4. If `w2_rome_repro.py` is not yet in the repository (e.g. not pushed from local), this notebook prompts you to upload the script and hparams instead.

> If the default Colab torch is already 2.9.x, no reinstall is needed; see section 5 (Notes) for version-conflict alternatives.


In [ ]:
# 1.1 Clone the project repository (robust: retry git clone, fallback to zip download)
import os, subprocess, sys, time, zipfile, urllib.request

REPO_URL = "https://github.com/FongChinWai-Vivi/CALM-Editor-CISC7021-Task1-Group26.git"
ZIP_URL = "https://github.com/FongChinWai-Vivi/CALM-Editor-CISC7021-Task1-Group26/archive/refs/heads/main.zip"
PROJ_DIR = "/content/CALM-Editor-CISC7021-Task1-Group26"
ZIP_PATH = "/content/repo.zip"
EXTRACT_DIR = "/content"
EXTRACTED_NAME = "CALM-Editor-CISC7021-Task1-Group26-main"

def clone_with_retry():
    # Try git clone up to 3 times with a 5s sleep between attempts
    for attempt in range(1, 4):
        try:
            print(f"Cloning project repository (attempt {attempt}/3) ...")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJ_DIR], check=True)
            return True
        except subprocess.CalledProcessError as e:
            print(f"git clone failed (attempt {attempt}/3): {e}")
            if attempt < 3:
                time.sleep(5)
    return False

def clone_via_zip():
    # Fallback: download the repository as a zip archive and extract it
    print("git clone failed after 3 attempts, falling back to zip download ...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print("Zip archive downloaded, extracting ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    extracted = os.path.join(EXTRACT_DIR, EXTRACTED_NAME)
    if os.path.exists(PROJ_DIR):
        print("Target directory already exists, keeping it.")
    else:
        os.rename(extracted, PROJ_DIR)
    print("Repository extracted from zip archive.")

if not os.path.isdir(PROJ_DIR):
    if not clone_with_retry():
        clone_via_zip()
else:
    print("Project directory already exists, skipping clone.")

print("Project directory:", PROJ_DIR)


In [ ]:

# 1.2 Check that the W2 reproduction script and GPT-J hparams are in place (may need to upload if not pushed locally)
script_path = os.path.join(PROJ_DIR, "EasyEdit", "examples", "w2_rome_repro.py")
hparams_path = os.path.join(PROJ_DIR, "EasyEdit", "hparams", "ROME", "gpt-j-6B.yaml")

if os.path.isfile(script_path) and os.path.isfile(hparams_path):
    print("w2_rome_repro.py and gpt-j-6B.yaml both exist, ready to run.")
else:
    print("The repository is missing the following files, please upload them:")
    print("  -", script_path)
    print("  -", hparams_path)
    print("Local paths:")
    print("  - EasyEdit/examples/w2_rome_repro.py")
    print("  - EasyEdit/hparams/ROME/gpt-j-6B.yaml")
    from google.colab import files
    up = files.upload()   # upload a local file
    import shutil
    for name in up.keys():
        if name.endswith("w2_rome_repro.py"):
            os.makedirs(os.path.dirname(script_path), exist_ok=True)
            shutil.move(name, script_path)
        elif name.endswith("gpt-j-6B.yaml"):
            os.makedirs(os.path.dirname(hparams_path), exist_ok=True)
            shutil.move(name, hparams_path)
    print("Upload complete.")


In [ ]:
# 1.3 Check torch version (skip reinstall by default, keep CUDA check)
import torch

print("Current torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Set to False only if you need to strictly align with EasyEdit's pinned torch==2.9.1
SKIP_TORCH_REINSTALL = True
MIN_TORCH = "2.9.0"
EXPECT_TORCH = "2.9.1"

def version_tuple(v):
    nums = []
    for x in v.split(".")[:3]:
        try:
            nums.append(int(x))
        except ValueError:
            break
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

cur = torch.__version__.split("+")[0]
if SKIP_TORCH_REINSTALL:
    if version_tuple(cur) >= version_tuple(MIN_TORCH):
        print(f"torch {cur} is compatible, skipping reinstall.")
    else:
        print(f"torch {cur} is older than {MIN_TORCH}; set SKIP_TORCH_REINSTALL=False to reinstall torch=={EXPECT_TORCH}.")
else:
    if cur != EXPECT_TORCH:
        print(f"Current torch {cur} != target {EXPECT_TORCH}, reinstalling cu126 build ...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
        print("torch reinstalled as 2.9.1 (cu126).")
        print("Please click Runtime -> Restart session, then Run All again (downloaded content is kept and skipped on re-run).")
        sys.exit(0)
    else:
        print("torch version satisfies the requirement, skipping reinstall.")


In [ ]:
# 1.4 Install EasyEdit dependencies (torch-related lines excluded to keep the existing torch)
import importlib, re

req_path = os.path.join(PROJ_DIR, "EasyEdit", "requirements.txt")
filtered_req = os.path.join(PROJ_DIR, "EasyEdit", "requirements_no_torch.txt")
with open(req_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Drop lines whose package name is torch / torchvision / torchaudio
kept = [ln for ln in lines if not re.match(r"^\s*(torch|torchvision|torchaudio)([=@<>\s]|$)", ln, re.IGNORECASE)]
with open(filtered_req, "w", encoding="utf-8") as f:
    f.writelines(kept)

print("Installing EasyEdit requirements with torch-related lines excluded ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", filtered_req], check=False)
print("requirements.txt installation finished (torch-related lines excluded, existing torch kept).")

# Fallback: ensure key packages are present
missing = []
for pkg in ["datasets", "huggingface_hub", "sentence_transformers", "einops", "higher", "omegaconf", "transformers"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)
if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

# Print key package versions
import torch, transformers, datasets
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("Environment setup complete.")



## 2. Download GPT-J-6B Weights (fp16, ~11.2GB)

- Use **ainize/gpt-j-6B-float16** (fp16 variant, verified to work locally);
- The official **EleutherAI/gpt-j-6b** only provides fp32 (~24GB), which does not fit in T4 VRAM and takes longer to download - **not recommended**;
- Weights are downloaded to `EasyEdit/hugging_cache/gpt-j-6B` (the script hparams point to this directory), with `allow_patterns` strictly filtering to keep only required files;
- Colab disk is about 78-100GB; check free space before downloading; on re-run after a disconnect the existing model is skipped.


In [ ]:

# 2.1 Check disk space
!df -h /content


In [ ]:

# 2.2 Download the model (idempotent: skip if already present)
from huggingface_hub import snapshot_download

model_dir = os.path.join(PROJ_DIR, "EasyEdit", "hugging_cache", "gpt-j-6B")

def has_model():
    if not os.path.isdir(model_dir):
        return False
    if any(f.endswith(".safetensors") for f in os.listdir(model_dir)):
        return True
    if os.path.isfile(os.path.join(model_dir, "pytorch_model.bin")):
        return True
    return os.path.isfile(os.path.join(model_dir, "pytorch_model.bin.index.json"))

if has_model():
    print("Model already exists, skipping download.")
else:
    print("Downloading ainize/gpt-j-6B-float16 (~11.2GB, 5-20 min depending on bandwidth)...")
    os.makedirs(model_dir, exist_ok=True)
    snapshot_download(
        repo_id="ainize/gpt-j-6B-float16",
        local_dir=model_dir,
        allow_patterns=["*.json", "*.txt", "*.safetensors", "*.bin", "*.md", "*.gitattributes"],
    )
    print("Model download complete.")
    # show downloaded files
    for f in sorted(os.listdir(model_dir))[:20]:
        print(" -", f)



## 3. Run ROME Editing (GPT-J-6B)

The script `EasyEdit/examples/w2_rome_repro.py` handles data download (official counterfact / zsre_mend_eval / MQuAKE) and metric evaluation. Supported arguments:

| Argument | Description | Notebook Default |
|---|---|---|
| `--model gpt-j-6B` | Use GPT-J-6B (fp16 weights) | Fixed |
| `--data` | counterfact / zsre / mquake | zsre |
| `--n` | Number of editing cases | 10 (smoke test), 50 for full |
| `--fp16` | Half precision (required on T4 16GB) | On |
| `--mom2-n-samples` | Number of samples for mom2 statistics (default 10000) | 1000 (faster) |

Reference: on a local RTX 4070 8GB one case takes ~570s (CPU rebatch bottleneck); Colab T4 16GB has ample VRAM and should be much faster. The first run includes data download and mom2 statistics, please be patient.


In [ ]:

# 3.1 Run ROME (adjust the parameters below as needed)
DATA_SUBSET   = "zsre"     # counterfact / zsre / mquake
N_CASES       = 10         # number of editing cases: 10 for smoke test, 50 for full
MOM2_SAMPLES  = 1000       # mom2 statistics sample count (default 10000; smaller is significantly faster)
USE_FP16      = True       # keep True on T4

cmd = [
    sys.executable, "examples/w2_rome_repro.py",
    "--model", "gpt-j-6B",
    "--data", DATA_SUBSET,
    "--n", str(N_CASES),
    "--mom2-n-samples", str(MOM2_SAMPLES),
]
if USE_FP16:
    cmd.append("--fp16")

print("Executing command:", " ".join(cmd))
print("Note: the first run includes data download + mom2 statistics; a single edit may take 5-30 min, keep the tab open.")
result = subprocess.run(cmd, cwd=os.path.join(PROJ_DIR, "EasyEdit"))
print("ROME run exit code:", result.returncode)



## 4. Save Results

The script writes `EasyEdit/results/eval/ROME_gpt-j-6B_<timestamp>.json`, containing per-case **reliability / generalisation / locality / portability** metrics.

This step copies the latest result to `/content/` and prints a metric summary; two save options are provided below (pick one or do both):
1. **Google Drive**: after mounting, copy to `MyDrive/CALM-Editor-W2/`;
2. **Direct download**: use the browser `files.download` to download to the local machine.


In [ ]:

# 4.1 Copy the latest result to /content and print a metric summary
import glob, json, shutil

res_dir = os.path.join(PROJ_DIR, "EasyEdit", "results", "eval")
jsons = sorted(glob.glob(os.path.join(res_dir, "ROME_gpt-j-6B_*.json")), key=os.path.getmtime)
if not jsons:
    raise FileNotFoundError("No result JSON found. Check whether the previous step ran successfully.")

latest = jsons[-1]
dst = "/content/ROME_gpt-j-6B_latest.json"
shutil.copy(latest, dst)
print("Latest result file:", os.path.basename(latest))
print("Copied to:", dst)

with open(latest, "r", encoding="utf-8") as f:
    data = json.load(f)

def get_metrics(d):
    if isinstance(d, list):
        d = {"post": d}
    if isinstance(d.get("post"), list) and d["post"]:
        first = d["post"][0]
        if isinstance(first, dict):
            return {k: first.get(k) for k in ("reliability", "generalisation", "locality", "portability")}
    return {k: v for k, v in d.items() if k in ("reliability", "generalisation", "locality", "portability")}

metrics = get_metrics(data)
print("\n===== Metric Summary =====")
found = False
for k, v in metrics.items():
    if v is not None:
        found = True
        print(f"  {k:15s}: {v}")
if not found:
    print("(Metric keys not found at top level / first case. Raw structure below for inspection)")
    print(json.dumps(data, ensure_ascii=False)[:2000])


In [ ]:

# 4.2 Option 1: Save to Google Drive (authorization prompt on first run)
from google.colab import drive
drive.mount("/content/drive")   # quickly skipped if already mounted

drive_dir = "/content/drive/MyDrive/CALM-Editor-W2"
os.makedirs(drive_dir, exist_ok=True)
shutil.copy(dst, os.path.join(drive_dir, os.path.basename(dst)))
print("Saved to:", os.path.join(drive_dir, os.path.basename(dst)))


In [ ]:

# 4.3 Option 2: Download directly to the local browser
from google.colab import files
files.download(dst)
print("Download triggered; if nothing happens, manually download from the left file panel: " + dst)



## 5. Notes

- **VRAM**: T4 16GB is enough for GPT-J-6B fp16 (~11.2GB weights + activations); do not try the fp32 24GB weights.
- **Session duration**: a free Colab session lasts up to ~12 hours, with weekly dynamic GPU quotas; ~90 minutes of inactivity causes a disconnect.
- **Recovery after disconnect**: reopen the notebook and **Run All** - the cached model and project on disk are automatically skipped; if a result JSON already exists, step 4 copies it and prints the summary.
- **Disk**: a standard Colab has ~78GB free; this task uses ~12GB (model + environment + data); avoid downloading other large models at the same time.
- **Version conflicts**: if installing `torch==2.9.1` from the cu126 index fails, use `--index-url https://download.pytorch.org/whl/cu124` instead, or allow other torch 2.x minor versions (ROME mainly requires `transformers==5.5.4` and is insensitive to torch minor versions).
- **Data**: zsRE / counterfact data are downloaded automatically by the script from memit.baulab.info (tens of MB on first run); MQuAKE is downloaded from GitHub. Internet access is required.
- **Metric meanings**: reliability=edit success rate, generalisation=rewrite generalisation, locality=preserving neighbouring facts, portability=cross-context transfer.
- **Parallel conflicts**: do not open multiple GPU sessions on the same account, or the quota may be exhausted and the T4 reclaimed.



## Done

After executing the notebook in order:
- The result JSON is located at `/content/ROME_gpt-j-6B_latest.json`;
- Optionally saved to `MyDrive/CALM-Editor-W2/` or downloaded locally;
- The metric summary was printed in the previous step.

To switch the data subset (counterfact / mquake) or increase the number of cases, adjust the parameters at the top of cell 3.1 and Run All again.
